<a href="https://colab.research.google.com/github/sweye/Sonara-llm-lyric/blob/main/ProjectSonara_LyricsEngine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎵 Project Sonara — Lyrics Engine
### Internal Inertiv AI Lyrics Fine-Tuning & Generation Pipeline

**Architecture:**
- **Model**: Llama 3 8B Instruct (4-bit quantized via Unsloth)
- **Method**: LoRA fine-tuning via PEFT + SFTTrainer (TRL)
- **Data**: Genius API → regex-cleaned artist lyrics → HF Dataset
- **UI**: Private Gradio dashboard with artist/era/style controls
- **Crash Safety**: Drive-backed checkpointing, resume-on-crash logic

---
**Run order**: Cell 0 (Drive mount) → Cell 1 (Install) → Cell 2 (Scrape) → Cell 3 (Model) → Cell 4 (Train) → Cell 5 (Export) → Cell 6 (UI)

## Cell 0 — Google Drive Mount & Directory Setup

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 0: Google Drive Mount + Crash-Safe Directory Setup
#
# WHY: Colab free/pro T4 sessions disconnect randomly. Saving LoRA adapter
# weights directly to Drive means even a full VM restart doesn't lose progress.
# All paths in this notebook flow through SONARA_ROOT.
# ─────────────────────────────────────────────────────────────────────────────

from google.colab import drive
import os

drive.mount('/content/drive')

# ── Base paths ────────────────────────────────────────────────────────────────
SONARA_ROOT       = '/content/drive/MyDrive/sonara_checkpoints'
LORA_ADAPTER_DIR  = os.path.join(SONARA_ROOT, 'sonara-lyrics-llama3-lora')  # final adapter
CKPT_DIR          = os.path.join(SONARA_ROOT, 'training_checkpoints')        # mid-training
DATASET_CACHE_DIR = os.path.join(SONARA_ROOT, 'dataset_cache')               # scraped lyrics
LOGS_DIR          = os.path.join(SONARA_ROOT, 'logs')

for d in [SONARA_ROOT, LORA_ADAPTER_DIR, CKPT_DIR, DATASET_CACHE_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f'✓  {d}')

print()
print('Drive mounted. All checkpoints will persist across session restarts.')
print(f'Root: {SONARA_ROOT}')

Mounted at /content/drive
✓  /content/drive/MyDrive/sonara_checkpoints
✓  /content/drive/MyDrive/sonara_checkpoints/sonara-lyrics-llama3-lora
✓  /content/drive/MyDrive/sonara_checkpoints/training_checkpoints
✓  /content/drive/MyDrive/sonara_checkpoints/dataset_cache
✓  /content/drive/MyDrive/sonara_checkpoints/logs

Drive mounted. All checkpoints will persist across session restarts.
Root: /content/drive/MyDrive/sonara_checkpoints


## Cell 1 — Environment & Tool Installation

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1: Install all dependencies
#
# UNSLOTH NOTE: We pull directly from the official GitHub repo to get the
# latest colab-new build. This avoids version mismatches with transformers
# that cause the bnb 4-bit quantization to silently break.
#
# TRL PICKLE BUG NOTE: Older TRL versions try to pickle the full model during
# mid-training saves, which crashes with Unsloth's custom quantized layers.
# Pinning trl>=0.8.6 and using save_strategy='steps' with only LoRA adapter
# saving avoids this entirely.
# ─────────────────────────────────────────────────────────────────────────────

import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

print('Step 1/5: Installing Unsloth (official GitHub build)...')
pip(
    'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git',
    '--no-deps'
)

print('Step 2/5: Installing HuggingFace stack + TRL...')
pip(
    'transformers>=4.40.0',
    'datasets>=2.18.0',
    'accelerate>=0.28.0',
    'peft>=0.10.0',
    'trl>=0.8.6',
    'bitsandbytes>=0.43.0',
    'sentencepiece',
    'protobuf',
)

print('Step 3/5: Installing xFormers (flash attention / memory efficiency)...')
pip('xformers', '--index-url', 'https://download.pytorch.org/whl/cu121')

print('Step 4/5: Installing Genius + Spotify scrapers...')
!pip('requests', 'python-dotenv', 'beautifulsoup4' 'syncedlyrics')

print('Step 5/5: Installing Gradio...')
!pip('gradio>=4.28.0')

print()
print('✓ All packages installed.')

# ── Quick sanity checks ────────────────────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Step 1/5: Installing Unsloth (official GitHub build)...
Step 2/5: Installing HuggingFace stack + TRL...
Step 3/5: Installing xFormers (flash attention / memory efficiency)...
Step 4/5: Installing Genius + Spotify scrapers...
/bin/bash: -c: line 1: syntax error near unexpected token `'requests','
/bin/bash: -c: line 1: `pip('requests', 'python-dotenv', 'beautifulsoup4' 'syncedlyrics')'
Step 5/5: Installing Gradio...
/bin/bash: -c: line 1: syntax error near unexpected token `'gradio>=4.28.0''
/bin/bash: -c: line 1: `pip('gradio>=4.28.0')'

✓ All packages installed.
PyTorch: 2.5.1+cu121
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## Cell 2 — Live Genius Scrape + Feature Eraser Pipeline

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2: Resilient Artist Pipeline (Powered by syncedlyrics - Musixmatch Bypassed)
# ─────────────────────────────────────────────────────────────────────────────

import re
import os
import json
import time
import random
import logging
import requests as _requests
import syncedlyrics
from tqdm.auto import tqdm

# Mutes all the library's warning prints (like 401s or missing tracks)
# so you only see your clean tqdm progress bar
logging.getLogger("syncedlyrics").setLevel(logging.CRITICAL)

# ─── API CREDENTIALS ──────────────────────────────────────────────────────────
GENIUS_ACCESS_TOKEN = 'tUo1B3C306DZMrhi60amCxHfAitBW89jdgte9pyQbAk4Rn6wWIdna5nh3EPmuCUe'

# ─── THE MASTER GENRE ARTIST MATRIX ───────────────────────────────────────────
GENRE_TARGETS = [
    {
        'tag': 'rap', 'status': 'Released', 'era': 'Hip-Hop and Rap',
        'voice_type': 'Dynamic rhythmic rap cadence, flow, and modern delivery styles',
        'max_songs_per_artist': 100,
        'artists': [
            'Playboi Carti', 'Travis Scott', 'Drake', 'Kendrick Lamar', 'J. Cole',
            'Future', 'Lil Baby', 'Gunna', 'Yeat', 'Ken Carson',
            'Destroy Lonely', 'Chief Keef', 'Pop Smoke', 'Central Cee', 'Lil Uzi Vert',
            '21 Savage', 'Kodak Black', 'Young Thug', 'Lil Wayne', 'Eminem',
            'Nas', 'Kanye West', 'A$AP Rocky', 'Tyler, The Creator',
            'Mac Miller', 'Juice WRLD', 'XXXTentacion', 'Lil Peep', 'Trippie Redd',
            'Ski Mask the Slump God', 'Denzel Curry', 'Vince Staples', 'Joey Bada$$', 'JID',
            'Cordae', 'Baby Keem', 'GloRilla', 'Megan Thee Stallion', 'Nicki Minaj',
            'Lil Durk', 'Polo G', 'Roddy Ricch', 'Don Toliver', 'NAV',
            'Lil Yachty', 'Ice Spice', 'Jack Harlow', 'Lil Tjay', 'NLE Choppa'
        ]
    },
    {
        'tag': 'rb', 'status': 'Released', 'era': 'R&B and Soul',
        'voice_type': 'Melodic, soulful vocal pacing and atmospheric rhythmic arrangements',
        'max_songs_per_artist': 100,
        'artists': [
            'The Weeknd', 'Frank Ocean', 'SZA', 'Bryson Tiller', 'PARTYNEXTDOOR',
            'Jhené Aiko', 'Summer Walker', 'Brent Faiyaz', 'Daniel Caesar', 'Miguel',
            'Chris Brown', 'Khalid', 'Kehlani', 'Trey Songz', 'Usher',
            'Alicia Keys', 'Beyoncé', 'Anderson .Paak', 'Steve Lacy', 'Syd',
            '6LACK', 'dvsn', 'Majid Jordan', 'Roy Woods', 'Kiana Ledé',
            'Tinashe', 'Ella Mai', 'Lucky Daye', 'Queen Naija', 'Victoria Monét',
            'Snoh Aalegra', 'Jazmine Sullivan', 'Erykah Badu', 'Lauryn Hill', 'Sade',
            'Blxst', 'Pink Sweat$', 'Tone Stith', 'Jacquees', 'Mario',
            'Omarion', 'Jeremih', 'Ty Dolla $ign', 'Leon Bridges', 'Teyana Taylor', 'Tink'
        ]
    },
    {
        'tag': 'pop', 'status': 'Released', 'era': 'Pop and Electronic',
        'voice_type': 'Polished vocal structures, memorable hooks, and modern pop arrangements',
        'max_songs_per_artist': 100,
        'artists': [
            'Taylor Swift', 'Billie Eilish', 'Ariana Grande', 'Dua Lipa', 'Olivia Rodrigo',
            'Justin Bieber', 'Ed Sheeran', 'Harry Styles', 'Bruno Mars', 'Rihanna',
            'Katy Perry', 'Lady Gaga', 'Miley Cyrus', 'Selena Gomez', 'Demi Lovato',
            'Shawn Mendes', 'Camila Cabello', 'Lorde', 'Lana Del Rey', 'Halsey',
            'Troye Sivan', 'Charli XCX', 'Caroline Polachek', 'Chappell Roan', 'Sabrina Carpenter',
            'Tate McRae', 'Gracie Abrams', 'Reneé Rapp', 'Benson Boone', 'Noah Kahan',
            'Hozier', 'Bleachers', 'The 1975', 'LANY', 'Lauv',
            'Chelsea Cutler', 'Jeremy Zucker', 'Conan Gray', 'girl in red', 'Clairo',
            'Omar Apollo', 'Dominic Fike', 'Remi Wolf', 'Rex Orange County', 'boygenius',
            'Phoebe Bridgers', 'Mitski', 'Weyes Blood', 'Maggie Rogers', 'Madison Beer'
        ]
    },
    {
        'tag': 'country', 'status': 'Released', 'era': 'Country and Folk',
        'voice_type': 'Storytelling vocal cadence, natural twang, and traditional structures',
        'max_songs_per_artist': 100,
        'artists': [
            'Morgan Wallen', 'Luke Combs', 'Zach Bryan', 'Chris Stapleton', 'Tyler Childers',
            'Cody Johnson', 'Lainey Wilson', 'Jelly Roll', 'Kacey Musgraves', 'Luke Bryan',
            'Jason Aldean', 'Blake Shelton', 'Carrie Underwood', 'Miranda Lambert', 'Eric Church',
            'Thomas Rhett', 'Kane Brown', 'HARDY', 'Bailey Zimmerman', 'Warren Zeiders',
            'Koe Wetzel', 'Colter Wall', 'Sturgill Simpson', 'Charley Crockett', 'Sierra Ferrell',
            'Billy Strings', 'Orville Peck', 'Maren Morris', 'Jon Pardi', 'Riley Green',
            'Parker McCollum', 'Jordan Davis', 'Cole Swindell', 'Walker Hayes', 'Dylan Scott',
            'Russell Dickerson', 'Brett Young', 'Chris Young', 'Sam Hunt', 'Keith Urban',
            'Brad Paisley', 'Tim McGraw', 'Kenny Chesney', 'George Strait', 'Willie Nelson',
            'Johnny Cash', 'Dolly Parton', 'Waylon Jennings', 'Hank Williams', 'Ernest'
        ]
    },
    {
        'tag': 'rock', 'status': 'Released', 'era': 'Rock and Metal',
        'voice_type': 'High-energy, gritty vocal performances and expressive delivery',
        'max_songs_per_artist': 100,
        'artists': [
            'Nirvana', 'Foo Fighters', 'Red Hot Chili Peppers', 'Green Day', 'Linkin Park',
            'My Chemical Romance', 'Fall Out Boy', 'Paramore', 'Blink-182', 'Yungblud',
            'Willow', 'Turnstile', 'Fontaines D.C.', 'Idles', 'Deftones',
            'Korn', 'Slipknot', 'System of a Down', 'Bring Me The Horizon', 'Bad Omens',
            'Sleep Token', 'Pierce the Veil', 'Sleeping with Sirens', 'All Time Low',
            'Mayday Parade', 'The Used', 'Yellowcard', 'Jimmy Eat World', 'Weezer',
            'Radiohead', 'Muse', 'Arctic Monkeys', 'The Killers', 'Cage the Elephant',
            'Tame Impala', 'MGMT', 'Foster the People', 'Twenty One Pilots', 'Panic! At The Disco',
            'Shinedown', 'Three Days Grace', 'Breaking Benjamin', 'Avenged Sevenfold', 'Metallica',
            'AC/DC', 'Led Zeppelin', 'Pink Floyd', 'Queens of the Stone Age', 'Royal Blood'
        ]
    }
]

# ─── GENIUS API INITIALIZATION FOR METADATA ONLY ──────────────────────────────
_api_session = _requests.Session()
_api_session.headers.update({
    'Authorization': f'Bearer {GENIUS_ACCESS_TOKEN}',
    'User-Agent':    'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'Accept':        'application/json',
})

def _api_get(path, params=None, retries=3):
    url = 'https://api.genius.com' + path
    for attempt in range(retries):
        try:
            r = _api_session.get(url, params=params, timeout=15)
            if r.status_code == 200: return r.json()
            elif r.status_code == 429: time.sleep(60 * (attempt + 1))
            else: return None
        except Exception: time.sleep(5)
    return None

def _resolve_artist_id(artist_name):
    data = _api_get('/search', params={'q': artist_name, 'per_page': 5})
    if not data: return None
    hits = data.get('response', {}).get('hits', [])
    for hit in hits:
        primary = hit.get('result', {}).get('primary_artist', {})
        if artist_name.lower() in primary.get('name', '').lower():
            return primary.get('id')
    return None

# ─── SYNCEDLYRICS INTEGRATION ────────────────────────────────────────────────
def _fetch_lyrics_synced(artist_name, track_name):
    """
    Bypasses Genius completely for lyrics. Uses syncedlyrics to search open
    databases. Skips Musixmatch to avoid 401 errors.
    """
    try:
        query = f"{track_name} {artist_name}"
        # We explicitly drop Musixmatch and only use open providers
        lyrics = syncedlyrics.search(
            query,
            providers=["Lrclib", "NetEase", "Megalobiz"]
        )
        return lyrics
    except Exception:
        return None

def clean_and_remove_features(raw_lyrics: str) -> str:
    """
    Cleans the raw text retrieved by syncedlyrics.
    Removes LRC timestamps like [01:23.45] and trims excess whitespace.
    """
    if not raw_lyrics: return ''

    # Remove timestamps
    raw_lyrics = re.sub(r'\[\d{2}:\d{2}\.\d{2,3}\]', '', raw_lyrics)
    # Remove blank lines left by timestamp removal
    raw_lyrics = re.sub(r'^\s+', '', raw_lyrics, flags=re.MULTILINE)

    return raw_lyrics.strip()

def build_instruction(artist, status, era, voice_type, title):
    return f"Write {status.lower()} lyrics for a track called '{title}' in the style of {artist}. Era: {era}. Vocal style: {voice_type}. Include structural section headers like [Verse 1], [Chorus], and [Outro]. Output only the lyrics with no commentary."

# ─── RUN ENGINE ─────────────────────────────────────────────────────────────
DATASET_CACHE_PATH = os.path.join(os.path.expanduser('~'), 'sonara_lyrics_dataset.json')
raw_records = []
completed_artists = set()

# Load existing data if restarting
if os.path.exists(DATASET_CACHE_PATH):
    try:
        with open(DATASET_CACHE_PATH, 'r') as f:
            raw_records = json.load(f)
            for record in raw_records:
                completed_artists.add(record['artist'])
    except Exception: pass

for target in GENRE_TARGETS:
    print(f'\n--- Pillar: {target["tag"].upper()} ---')
    for artist_name in target['artists']:
        if artist_name in completed_artists: continue

        _artist_id = _resolve_artist_id(artist_name)
        if not _artist_id: continue

        print(f"🚀 Fetching: {artist_name}")
        songs_list = []
        page = 1

        while len(songs_list) < target['max_songs_per_artist']:
            data = _api_get(f'/artists/{_artist_id}/songs', params={'sort': 'popularity', 'per_page': 50, 'page': page})
            if not data or not data.get('response', {}).get('songs'): break
            songs_list.extend(data['response']['songs'])
            if len(data['response']['songs']) < 50: break
            page += 1

        # Use tqdm to show a progress bar per artist
        for song in tqdm(songs_list, desc=f'  {artist_name}', leave=False):
            # No heavy sleep needed anymore since we aren't scraping one domain
            time.sleep(random.uniform(0.1, 0.5))

            # Use syncedlyrics to fetch the text (Musixmatch skipped)
            lyrics = _fetch_lyrics_synced(artist_name, song.get('title'))

            if not lyrics:
                continue

            clean_lyrics = clean_and_remove_features(lyrics)

            # Skip if result is too short (likely a bad scrape or instrumental)
            if len(clean_lyrics.split()) < 50:
                continue

            raw_records.append({
                'instruction': build_instruction(artist_name, target['status'], target['era'], target['voice_type'], song['title']),
                'output': clean_lyrics,
                'artist': artist_name
            })

        completed_artists.add(artist_name)

        # Save cache after every artist to prevent data loss
        with open(DATASET_CACHE_PATH, 'w') as f:
            json.dump(raw_records, f)

print(f'\n🏁 Pipeline Run Concluded. {len(raw_records)} total records.')


--- Pillar: RAP ---
🚀 Fetching: Playboi Carti


  Playboi Carti:   0%|          | 0/99 [00:00<?, ?it/s]

🚀 Fetching: Travis Scott


  Travis Scott:   0%|          | 0/100 [00:00<?, ?it/s]

🚀 Fetching: Drake


...

  Kanye West:   0%|          | 0/100 [00:00<?, ?it/s]

🚀 Fetching: A$AP Rocky


  A$AP Rocky:   0%|          | 0/100 [00:00<?, ?it/s]

🚀 Fetching: Tyler, The Creator


  Tyler, The Creator:   0%|          | 0/100 [00:00<?, ?it/s]

In [7]:
!pip install syncedlyrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 82.9 MB/s eta 0:00:00


## Cell 3 — Model Initialization, LoRA Setup & Token Structuring

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3: Load Llama 3 8B Instruct → 4-bit quant → LoRA adapters → tokenize
#
# MODEL CHOICE:
#   unsloth/llama-3-8b-Instruct-bnb-4bit is the pre-quantized HF version.
#   Loading it with Unsloth's FastLanguageModel gives ~2× faster training
#   than native HF + bitsandbytes by fusing attention kernels.
#
# LORA TARGET RATIONALE:
#   We target all 7 projection layers (q,k,v,o + gate,up,down in FFN).
#   This gives maximum expressivity for style transfer. For a lighter
#   adapter that still captures rhyme/cadence, you can drop gate/up/down
#   and halve the r value.
#
# TEMPLATE:
#   Llama 3 Instruct uses a specific chat template with special tokens:
#   <|begin_of_text|> ... <|start_header_id|>user<|end_header_id|> ...
#   We MUST use this exact template or the model ignores our fine-tuning.
# ─────────────────────────────────────────────────────────────────────────────

from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048   # max tokens per training sample (lyrics rarely exceed this)
DTYPE          = None   # None = auto-detect (float16 on T4, bfloat16 on A100)
LOAD_IN_4BIT   = True   # 4-bit quantization = ~4× less VRAM vs full precision

print('Loading Llama 3 8B Instruct (4-bit)...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'unsloth/llama-3-8b-Instruct-bnb-4bit',
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = DTYPE,
    load_in_4bit   = LOAD_IN_4BIT,
    # token = 'hf_...'  ← add HF token here if you hit rate limits
)
print('✓ Base model loaded.')

# ─── PEFT LoRA INJECTION ───────────────────────────────────────────────────────
print('Injecting LoRA adapters...')
model = FastLanguageModel.get_peft_model(
    model,
    r                      = 16,       # rank: 8-16 is sweet spot for style transfer
    target_modules         = [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',   # attention projections
        'gate_proj', 'up_proj', 'down_proj',        # FFN projections
    ],
    lora_alpha             = 16,       # scaling factor (keep equal to r)
    lora_dropout           = 0.05,     # small dropout prevents overfit on small lyric corpus
    bias                   = 'none',
    use_gradient_checkpointing = 'unsloth',  # Unsloth's custom GC — 30% less VRAM
    random_state           = 42,
    use_rslora             = False,    # rank-stabilized LoRA — enable for r≥64
    loftq_config           = None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'✓ LoRA injected.')
print(f'   Trainable params: {trainable:,} ({100*trainable/total:.2f}% of {total:,} total)')


# ─── LLAMA 3 INSTRUCT TEMPLATE ─────────────────────────────────────────────────
# This is the OFFICIAL Llama 3 Instruct chat format.
# We inject {instruction} into the user turn and {output} into the assistant turn.
# The EOS token <|eot_id|> at the end of each turn signals turn boundaries.
# DO NOT change this template — the model was pre-trained with these exact tokens.

LLAMA3_TEMPLATE = (
    '<|begin_of_text|>'
    '<|start_header_id|>user<|end_header_id|>\n'
    '{instruction}'
    '<|eot_id|>'
    '<|start_header_id|>assistant<|end_header_id|>\n'
    '{output}'
    '<|eot_id|>'
)

EOS_TOKEN = tokenizer.eos_token  # typically <|eot_id|> for Llama 3 Instruct


def format_sample(sample):
    """
    Maps a dataset row into a single 'text' field using the Llama 3 template.
    SFTTrainer will train on the full sequence (instruction + output).
    """
    text = LLAMA3_TEMPLATE.format(
        instruction = sample['instruction'],
        output      = sample['output'],
    )
    return {'text': text}


# ── Apply template to entire dataset ─────────────────────────────────────────
formatted_dataset = hf_dataset.map(format_sample, desc='Applying Llama 3 template')

print(f'\n✓ Dataset tokenized. {len(formatted_dataset)} samples ready for training.')
print()
print('--- Formatted sample preview (first 600 chars) ---')
print(formatted_dataset[0]['text'][:600] if len(formatted_dataset) > 0 else '(empty)')

## Cell 4 — Safe SFT Training Loop
**Crash-safe**: LoRA adapters saved to Drive every 50 steps. Resumable.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4: SFTTrainer Configuration + Training
#
# ANTI-CRASH STRATEGY:
#   1. output_dir points to Google Drive (CKPT_DIR from Cell 0)
#   2. save_strategy='steps' + save_steps=50 saves lightweight LoRA layers
#      every 50 gradient updates — not the full model, just the adapter delta
#   3. save_total_limit=2 keeps only the 2 most recent checkpoints to avoid
#      filling your 15GB Drive quota
#   4. If the session crashes: reconnect, re-run Cells 0-3, then call
#      trainer.train(resume_from_checkpoint=True) — see RESUME block below
#
# PICKLE BUG FIX:
#   The TRL < 0.8.6 SFTTrainer tried to serialize the full model during saves,
#   which failed on Unsloth's custom quantized layer types.
#   With TRL >= 0.8.6 + Unsloth's PEFT model, only the LoRA adapter tensors
#   (adapter_model.safetensors) are saved — no pickle, no crash.
# ─────────────────────────────────────────────────────────────────────────────

from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# ── Training hyperparameters ───────────────────────────────────────────────────
TRAIN_EPOCHS         = 3       # 3 epochs is usually enough for lyric style capture
BATCH_SIZE           = 2       # safe for T4 16GB — increase to 4 on A100
GRAD_ACCUM           = 4       # effective batch = 2 * 4 = 8
LR                   = 2e-4
WARMUP_RATIO         = 0.05    # 5% warmup over total steps
WEIGHT_DECAY         = 0.01
SAVE_STEPS           = 50      # save LoRA checkpoint every 50 steps to Drive
LOGGING_STEPS        = 10
SAVE_TOTAL_LIMIT     = 2       # keep only last 2 checkpoints on Drive

training_args = TrainingArguments(
    # ── Output / checkpoint dirs (Drive-backed) ──────────────────────────────
    output_dir          = CKPT_DIR,               # → /content/drive/MyDrive/sonara_checkpoints/training_checkpoints
    overwrite_output_dir = True,

    # ── Batch / accumulation ─────────────────────────────────────────────────
    per_device_train_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps  = GRAD_ACCUM,
    num_train_epochs             = TRAIN_EPOCHS,

    # ── Optimization ─────────────────────────────────────────────────────────
    learning_rate          = LR,
    warmup_ratio           = WARMUP_RATIO,
    weight_decay           = WEIGHT_DECAY,
    lr_scheduler_type      = 'cosine',
    optim                  = 'adamw_8bit',       # 8-bit AdamW = less VRAM than fp32 Adam

    # ── Precision ────────────────────────────────────────────────────────────
    fp16    = not is_bfloat16_supported(),        # fp16 on T4
    bf16    = is_bfloat16_supported(),            # bf16 on A100/H100

    # ── Checkpoint strategy (ANTI-CRASH) ─────────────────────────────────────
    save_strategy   = 'steps',                   # NOT 'no' — lightweight LoRA-only saves
    save_steps      = SAVE_STEPS,                # save to Drive every 50 steps
    save_total_limit = SAVE_TOTAL_LIMIT,         # cap Drive usage at 2 checkpoints

    # ── Logging ──────────────────────────────────────────────────────────────
    logging_steps   = LOGGING_STEPS,
    logging_dir     = LOGS_DIR,
    report_to       = 'none',                    # set 'tensorboard' if you want live graphs

    # ── Misc ──────────────────────────────────────────────────────────────────
    seed               = 42,
    dataloader_num_workers = 2,
    gradient_checkpointing = True,
)

trainer = SFTTrainer(
    model            = model,
    tokenizer        = tokenizer,
    train_dataset    = formatted_dataset,
    dataset_text_field = 'text',              # column containing the formatted samples
    max_seq_length   = MAX_SEQ_LENGTH,
    args             = training_args,
    packing          = True,                  # pack multiple short lyrics into one context window
)

# ── RESUME BLOCK ─────────────────────────────────────────────────────────────
# If your session crashed, re-run Cells 0-3 to rebuild model + dataset,
# then UNCOMMENT the line below and run this cell. The trainer will find the
# most recent checkpoint in CKPT_DIR and pick up exactly where it stopped.
#
#   trainer.train(resume_from_checkpoint=True)
#
# For a normal (fresh) run, use the line below:

print('Starting training...')
print(f'  Output dir (Drive): {CKPT_DIR}')
print(f'  Save every {SAVE_STEPS} steps | Keep last {SAVE_TOTAL_LIMIT} checkpoints')
print(f'  Effective batch: {BATCH_SIZE * GRAD_ACCUM} | Epochs: {TRAIN_EPOCHS}')
print()

train_result = trainer.train()   # ← swap this for trainer.train(resume_from_checkpoint=True) after a crash

print()
print('✓ Training complete.')
print(f'  Final loss: {train_result.training_loss:.4f}')
print(f'  Total steps: {train_result.global_step}')

## Cell 5 — Manual LoRA Weight Export to Drive

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5: Explicit adapter export
#
# WHY MANUAL SAVE:
#   Even though we have step-based saving enabled, those mid-training
#   checkpoints are in HF Trainer format (with optimizer states, etc.).
#   This cell does a clean final export of ONLY the LoRA adapter weights
#   (adapter_model.safetensors + adapter_config.json) to LORA_ADAPTER_DIR.
#   The Gradio UI in Cell 6 loads from exactly this directory.
#
# OUTPUT FILES:
#   adapter_model.safetensors  — the trained LoRA delta weights
#   adapter_config.json        — LoRA hyperparams (r, alpha, target_modules)
#   tokenizer.json             — tokenizer vocab
#   tokenizer_config.json      — tokenizer settings + Llama 3 chat template
#   special_tokens_map.json    — special token definitions
# ─────────────────────────────────────────────────────────────────────────────

import os

print(f'Exporting LoRA adapter to: {LORA_ADAPTER_DIR}')

# Save adapter weights (safetensors format — no pickle)
model.save_pretrained(LORA_ADAPTER_DIR)
print(f'  ✓ adapter_model.safetensors saved')

# Save tokenizer (includes the Llama 3 Instruct chat template)
tokenizer.save_pretrained(LORA_ADAPTER_DIR)
print(f'  ✓ tokenizer files saved')

# List exported files for confirmation
exported = os.listdir(LORA_ADAPTER_DIR)
print(f'\nFiles in {LORA_ADAPTER_DIR}:')
for f in sorted(exported):
    size = os.path.getsize(os.path.join(LORA_ADAPTER_DIR, f))
    print(f'  {f:<45}  {size/1e6:.1f} MB')

print()
print('Export complete. Run Cell 6 to launch the Gradio UI.')

## Cell 6 — Private Gradio Lyrics Dashboard UI

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6: Gradio Dashboard — Project Sonara Lyrics Generator
#
# This cell:
#   1. Reloads the trained LoRA adapter from LORA_ADAPTER_DIR
#   2. Builds a Gradio Blocks UI with full generation controls
#   3. Launches with share=True → gives you a public xxxx.gradio.live URL
#      that anyone with the link can use (no auth by default)
#
# TO ADD AUTH: replace launch(...) with
#   demo.launch(share=True, auth=('sonara', 'your_password_here'))
# ─────────────────────────────────────────────────────────────────────────────

import gradio as gr
import torch
from unsloth import FastLanguageModel
from transformers import TextIteratorStreamer
from threading import Thread

# ─── Reload model in inference mode ──────────────────────────────────────────
print('Loading trained adapter for inference...')

inf_model, inf_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = LORA_ADAPTER_DIR,   # loads base model + LoRA adapter
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)
FastLanguageModel.for_inference(inf_model)   # Unsloth inference mode: 2× faster

print('✓ Model ready for inference.')


# ─── Artist metadata pulled from ARTIST_TARGETS ──────────────────────────────
# Build a lookup dict so the UI dropdowns auto-populate with the artists
# and eras you actually trained on.
ARTIST_META = {
    t['artist']: {
        'eras':       [t['era']],
        'voice_type': t['voice_type'],
        'status':     t['status'],
    }
    for t in ARTIST_TARGETS
}
ARTIST_NAMES = list(ARTIST_META.keys())


# ─── Inference function ───────────────────────────────────────────────────────
def generate_lyrics(
    artist: str,
    release_status: str,
    custom_era: str,
    custom_voice: str,
    extra_prompt: str,
    temperature: float,
    max_new_tokens: int,
):
    """
    Runs inference with the fine-tuned model.
    Returns a generator that yields tokens one by one for streaming display.
    """
    meta       = ARTIST_META.get(artist, {})
    era        = custom_era.strip()   or (meta.get('eras', ['Unknown'])[0])
    voice_type = custom_voice.strip() or meta.get('voice_type', 'melodic trap')
    status     = release_status

    # Build instruction — same format used during training
    instruction = (
        f"Write {status.lower()} rap lyrics in the style of {artist}. "
        f"Era: {era}. "
        f"Vocal style: {voice_type}. "
        f"Include structural section headers like [Verse 1], [Chorus], and [Outro]. "
        f"Match the cadence, rhyme density, slang, and energy of {artist}'s {era} era. "
    )
    if extra_prompt.strip():
        instruction += f"Additional direction: {extra_prompt.strip()}. "
    instruction += "Output only the lyrics with no commentary, no titles, no explanations."

    # Format with Llama 3 template (inference: no output side)
    prompt = (
        '<|begin_of_text|>'
        '<|start_header_id|>user<|end_header_id|>\n'
        f'{instruction}'
        '<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n'
    )

    inputs = inf_tokenizer(
        prompt,
        return_tensors='pt',
        add_special_tokens=False
    ).to(inf_model.device)

    # Streaming setup
    streamer = TextIteratorStreamer(
        inf_tokenizer,
        skip_prompt=True,
        skip_special_tokens=True
    )

    generate_kwargs = {
        **inputs,
        'streamer':       streamer,
        'max_new_tokens': int(max_new_tokens),
        'temperature':    float(temperature),
        'do_sample':      True,
        'top_p':          0.92,
        'repetition_penalty': 1.15,   # reduce chorus-style line repetition
        'use_cache':      True,
    }

    thread = Thread(target=inf_model.generate, kwargs=generate_kwargs)
    thread.start()

    generated = ''
    for token in streamer:
        generated += token
        yield generated   # Gradio streams partial output live


# ─── Gradio UI ────────────────────────────────────────────────────────────────
SONARA_CSS = """
/* Project Sonara — dark theme overrides */
body, .gradio-container { background-color: #0e0e12 !important; }
.block, .panel, .gr-box { background-color: #1a1a24 !important; border: 1px solid #2e2e40 !important; }
label, .label-wrap span, h1, h2, h3, p { color: #c8c8dd !important; }
.gr-button-primary { background: linear-gradient(135deg, #7b2fff 0%, #4f46e5 100%) !important;
                     color: #fff !important; border: none !important; font-weight: 700; }
.gr-button-secondary { background: #2a2a3a !important; color: #c8c8dd !important; }
input, textarea, select { background-color: #1e1e2e !important;
                           color: #e0e0f0 !important; border-color: #3a3a50 !important; }
"""

with gr.Blocks(theme=gr.themes.Soft(), css=SONARA_CSS, title='Project Sonara') as demo:

    gr.HTML("""
        <div style='text-align:center; padding: 20px 0 10px;'>
            <h1 style='font-size:2.2em; font-weight:800; color:#a78bfa;
                       letter-spacing:0.05em; margin:0;'>🎵 Project Sonara</h1>
            <p style='color:#888;margin:6px 0 0;font-size:0.95em;'>
                Inertiv Internal Lyrics Engine — Fine-tuned Llama 3 8B
            </p>
        </div>
    """)

    with gr.Row():

        # ── LEFT COLUMN: Controls ───────────────────────────────────────────
        with gr.Column(scale=1, min_width=320):
            gr.Markdown('### Artist & Style Controls')

            inp_artist = gr.Dropdown(
                choices   = ARTIST_NAMES,
                value     = ARTIST_NAMES[0] if ARTIST_NAMES else None,
                label     = 'Artist',
                info      = 'Select which artist style to generate in',
            )

            inp_status = gr.Radio(
                choices = ['Released', 'Unreleased'],
                value   = 'Released',
                label   = 'Catalog Tracking',
                info    = 'Mark the generated track as Released or Unreleased',
            )

            inp_era = gr.Textbox(
                label       = 'Era Override',
                placeholder = 'e.g. Whole Lotta Red, Die Lit, Rodeo...',
                info        = 'Leave blank to use the default era for this artist',
            )

            inp_voice = gr.Textbox(
                label       = 'Vocal Style Override',
                placeholder = 'e.g. aggressive double-time, melodic sing-rap...',
                info        = 'Leave blank to use the trained vocal profile',
            )

            inp_extra = gr.Textbox(
                label       = 'Custom Direction (optional)',
                placeholder = 'e.g. Write about loss, set in a dark club...',
                lines       = 3,
                info        = 'Any extra thematic or contextual instruction',
            )

            inp_temp = gr.Slider(
                minimum = 0.3,
                maximum = 1.4,
                value   = 0.85,
                step    = 0.05,
                label   = 'Creativity / Temperature',
                info    = 'Lower = tighter style match. Higher = more experimental.',
            )

            inp_tokens = gr.Slider(
                minimum = 128,
                maximum = 1024,
                value   = 512,
                step    = 64,
                label   = 'Max New Tokens',
                info    = 'Controls approximate lyric length',
            )

            gr.Markdown('---')
            btn_generate = gr.Button('🎵 Generate Lyrics', variant='primary', size='lg')
            btn_clear    = gr.Button('Clear', variant='secondary')

        # ── RIGHT COLUMN: Output ────────────────────────────────────────────
        with gr.Column(scale=2, min_width=480):
            gr.Markdown('### Generated Lyrics')

            out_lyrics = gr.Textbox(
                label     = 'Output',
                lines     = 32,
                max_lines = 60,
                show_copy_button = True,
                placeholder = 'Your feature-stripped, structured lyric output will appear here...',
                interactive = False,
            )

            gr.HTML("""
                <div style='color:#555;font-size:0.8em;margin-top:8px;'>
                    ⚠ Internal use only — Inertiv / Project Sonara
                </div>
            """)

    # ── Event wiring ──────────────────────────────────────────────────────────
    btn_generate.click(
        fn      = generate_lyrics,
        inputs  = [inp_artist, inp_status, inp_era, inp_voice,
                   inp_extra, inp_temp, inp_tokens],
        outputs = out_lyrics,
        api_name = 'generate',
    )

    btn_clear.click(
        fn      = lambda: '',
        inputs  = [],
        outputs = out_lyrics,
    )

    # Keyboard shortcut: Ctrl+Enter in any input box triggers generation
    for inp in [inp_extra, inp_era, inp_voice]:
        inp.submit(
            fn      = generate_lyrics,
            inputs  = [inp_artist, inp_status, inp_era, inp_voice,
                       inp_extra, inp_temp, inp_tokens],
            outputs = out_lyrics,
        )


# ─── Launch ───────────────────────────────────────────────────────────────────
# share=True → generates a public xxxx.gradio.live URL valid for 72 hours.
# Add auth=(username, password) to restrict access:
#   demo.launch(share=True, auth=('sonara', 'your_secret_password'))

print('Launching Gradio UI...')
demo.launch(
    share          = True,
    show_error     = True,
    quiet          = False,
    inbrowser      = False,   # set True if you want auto-open (doesn't work in Colab)
)